# Supplementary Table 6

Shuffled-query controls: for each of 60 acquisitions and two shuffle seeds, the number of canonical peptide observations accepted at peptide q ≤ 0.01 and the number of protein entries selected, for the exact, bag, mass and combined lookup routes. This notebook rebuilds the table from `source_data/STab6/`, compares it with the table as published, repeats the producer's checks and prints the numbers quoted in the legend and the Supplementary Methods.

Run from the repository root: `python paper.py build STab6`.

In [1]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()

## Table

480 acquisition-seed rows (60 acquisitions x 2 seeds x 4 routes) and a four-row summary per route. Supports the legend of Supplementary Table 6 (v28d line 485) and the sentence "Shuffled-query controls yielded no accepted peptides through the exact or bag route and 13,512 peptide–acquisition–seed observations through the mass route across 60 acquisitions and two seeds" (v28d line 561). Unit: acquisition-seed run; n = 60 acquisitions (15 CAMPI, 43 healthy stool, 1 CapScan, 1 PREDICT), 2 seeds each. No test; counts are sums over runs and are not globally distinct sequences.

In [2]:
import pandas as pd
d = panels.load(SD)
acq = panels.build_acquisition_seed_rows(d)
summary = panels.build_summary(acq)
paths = panels.write_table(acq, summary, OUT)
print(acq.shape)
print(summary.to_string())
print(panels.cohort_mix(d).to_string())
acq.head(8)

(480, 13)
          accepted  selected_proteins
method                               
bags             0              78239
combined     13518             561331
exact            0              55229
mass         13512             561126
cohort
HSTOOL     43
CAMPI      15
CAPSCAN     1
PREDICT     1


,receipt,cohort,seed,method,q,level,accepted,exact_accepted,gained,lost,net,ratio_to_exact,selected_proteins
0,S01,CAMPI,1,exact,0.01,canonical_peptide,0,0,0,0,0,NaN,1012
1,S01,CAMPI,1,bags,0.01,canonical_peptide,0,0,0,0,0,NaN,1659
2,S01,CAMPI,1,mass,0.01,canonical_peptide,0,0,0,0,0,NaN,4724
3,S01,CAMPI,1,combined,0.01,canonical_peptide,0,0,0,0,0,NaN,4732
4,S01,CAMPI,2,exact,0.01,canonical_peptide,0,0,0,0,0,NaN,1027
5,S01,CAMPI,2,bags,0.01,canonical_peptide,0,0,0,0,0,NaN,1662
6,S01,CAMPI,2,mass,0.01,canonical_peptide,0,0,0,0,0,NaN,4767
7,S01,CAMPI,2,combined,0.01,canonical_peptide,0,0,0,0,0,NaN,4779


## Producer checks

The producer's own checks, repeated on the source tables: 60 receipts of 80 rows; every receipt complete and labelled workflow FDR unverified; the accounting identities of `finish.py`; selected protein counts independent of q and level; 12 Sage result files per receipt; the `package_release.py` assertions on the mass and exact totals.

In [3]:
chk, ok = panels.checks(d)
chk.to_csv(OUT / "CHECKS.tsv", sep="\t", index=False)
print(chk.to_string(index=False))
print("ALL_CHECKS_PASS" if ok else "CHECK_FAILURE")
assert ok

                                              check                         value  pass
                                           receipts                            60  True
                         rows per receipt (min,max)                      (80, 80)  True
                                              seeds                        [1, 2]  True
                                            methods [bags, combined, exact, mass]  True
status all NULL_DIAGNOSTIC_COMPLETE_REVIEW_REQUIRED                            60  True
                        workflow_fdr all UNVERIFIED                            60  True
                   net == accepted - exact_accepted                          4800  True
                               net == gained - lost                          4800  True
        selected_proteins constant over q and level                             1  True
            Sage result files per receipt (min,max)                      (12, 12)  True
                    acquisition-

## Checks against the table as produced and recomputed numbers

The two TSVs must be byte-identical to `expected/`; the workbook is compared cell by cell (its bytes carry a timestamp). Receipts go to `checks/`. Then every number in `NUMBERS.tsv` is printed in order.

In [4]:
import hashlib, json
def sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()
exp = UNIT / "expected"
receipt = {}
for k, name in (("by_acq", panels.TSV_BY_ACQ), ("summary", panels.TSV_SUMMARY)):
    receipt[name] = {"new_sha256": sha(paths[k]), "expected_sha256": sha(exp / name)}
    receipt[name]["identical"] = receipt[name]["new_sha256"] == receipt[name]["expected_sha256"]
receipt[panels.XLSX] = panels.compare_xlsx_values(paths["xlsx"], exp / panels.XLSX)
receipt[panels.XLSX]["expected_sha256"] = sha(exp / panels.XLSX)
receipt["result"] = "IDENTICAL" if all(v["identical"] for v in receipt.values()) else "DIFFERENT"
(UNIT / "checks").mkdir(exist_ok=True)
(UNIT / "checks" / "table.json").write_text(json.dumps(receipt, indent=2) + "\n")
print(json.dumps(receipt, indent=2))
assert receipt["result"] == "IDENTICAL"

print("NUMBERS.tsv order")
print("485 acquisitions:", panels.stat_n_acquisitions(d))
print("485 seeds:", panels.word(panels.stat_n_seeds(d)))
print("485 q cutoff:", panels.stat_q_cutoff(d))
print("485 exact accepted:", panels.word(panels.stat_accepted(d, "exact")))
print("485 bags accepted:", panels.word(panels.stat_accepted(d, "bags")))
print("485 mass accepted:", panels.stat_accepted(d, "mass"))
print("485 combined accepted:", panels.stat_accepted(d, "combined"))
for m in panels.METHODS:
    print(f"485 {m} selected protein entries:", panels.stat_selected_proteins(d, m))
print("561 mass accepted:", panels.stat_accepted(d, "mass"))
print("561 acquisitions:", panels.stat_n_acquisitions(d))
print("561 seeds:", panels.word(panels.stat_n_seeds(d)))

{
  "null_control_by_acquisition_seed.tsv": {
    "new_sha256": "3efbcb3f3dd8a19c141c836727bae8a2ee6676a8124b72b3fd580008c173e896",
    "expected_sha256": "3efbcb3f3dd8a19c141c836727bae8a2ee6676a8124b72b3fd580008c173e896",
    "identical": true
  },
  "null_control_summary.tsv": {
    "new_sha256": "e794df7b7cbf3a57004123ee7b09405089c1f8a4b2d4cd942e1731bcdf720e2c",
    "expected_sha256": "e794df7b7cbf3a57004123ee7b09405089c1f8a4b2d4cd942e1731bcdf720e2c",
    "identical": true
  },
  "Supplementary_Table_6.xlsx": {
    "identical": true,
    "sheets": [
      "Read me",
      "Summary",
      "Acquisition_seed_rows"
    ],
    "differences": [],
    "expected_sha256": "11f7be117046d8eef2361ea05c2c710bed8728fa4587a96328bd6e9d74529ceb"
  },
  "result": "IDENTICAL"
}
NUMBERS.tsv order
485 acquisitions: 60
485 seeds: two
485 q cutoff: 0.01
485 exact accepted: zero
485 bags accepted: zero
485 mass accepted: 13512
485 combined accepted: 13518
485 exact selected protein entries: 55229
485 bags